# Decision Trees

----

In [36]:
# Import libraries
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.inspection import PartialDependenceDisplay

## Read in data and perform additional cleaning

In [37]:
df = pd.read_csv("GamingStudy_data_clean.csv")

# Remove observations missing SPIN_T
df = df[df["SPIN_T"].notna()].copy()

df.head()

,S. No.,Timestamp,GAD1,GAD2,GAD3,GAD4,GAD5,GAD6,GAD7,GADE,...,Birthplace,Residence,Reference,Playstyle,accept,GAD_T,SWL_T,SPIN_T,Residence_ISO3,Birthplace_ISO3
0,1,42052.00437,0,0,0,0,1,0,0,Not difficult at all,...,USA,USA,Reddit,Singleplayer,Accept,1,23,5.0,USA,USA
1,2,42052.00680,1,2,2,2,0,1,0,Somewhat difficult,...,USA,USA,Reddit,Multiplayer - online - with strangers,Accept,8,16,33.0,USA,USA
2,3,42052.03860,0,2,2,0,0,3,1,Not difficult at all,...,Germany,Germany,Reddit,Singleplayer,Accept,8,17,31.0,DEU,DEU
3,4,42052.06804,0,0,0,0,0,0,0,Not difficult at all,...,USA,USA,Reddit,Multiplayer - online - with online acquaintanc...,Accept,0,17,11.0,USA,USA
4,5,42052.08948,2,1,2,2,2,3,2,Very difficult,...,USA,South Korea,Reddit,Multiplayer - online - with strangers,Accept,14,14,13.0,KOR,USA


In [38]:
df.shape

(12813, 55)

In [39]:
df["accept"].value_counts(dropna=False)

accept
Accept    12431
NaN         382
Name: count, dtype: int64

In [40]:
# generate column names to remove
gad_cols = [f"GAD{i}" for i in range(1, 8)]
swl_cols = [f"SWL{i}" for i in range(1, 6)]
spin_cols = [f"SPIN{i}" for i in range(1, 18)]

drop_cols = (
    ["GAD_T", "S. No.", "Timestamp",
     "Residence_ISO3", "Birthplace_ISO3",
     "accept", "GADE"]
    + gad_cols
    + swl_cols
    + spin_cols
)

# define target variable
y = df["GAD_T"]

# define feature variables (remove GADE yes or no?) (keep only SPIN_T and SWL_T?)
X = df.drop(columns=drop_cols)

X.head()

,Game,Platform,Hours,earnings,whyplay,League,highestleague,streams,Narcissism,Gender,Age,Work,Degree,Birthplace,Residence,Reference,Playstyle,SWL_T,SPIN_T
0,Skyrim,"Console (PS, Xbox, ...)",15.0,I play for fun,having fun,NaN,NaN,0.0,1.0,Male,25,Unemployed / between jobs,Bachelor (or equivalent),USA,USA,Reddit,Singleplayer,23,5.0
1,Other,PC,8.0,I play for fun,having fun,NaN,NaN,2.0,1.0,Male,41,Unemployed / between jobs,Bachelor (or equivalent),USA,USA,Reddit,Multiplayer - online - with strangers,16,33.0
2,Other,PC,0.0,I play for fun,having fun,NaN,NaN,0.0,4.0,Female,32,Employed,Bachelor (or equivalent),Germany,Germany,Reddit,Singleplayer,17,31.0
3,Other,PC,20.0,I play for fun,improving,NaN,NaN,5.0,2.0,Male,28,Employed,Bachelor (or equivalent),USA,USA,Reddit,Multiplayer - online - with online acquaintanc...,17,11.0
4,Other,"Console (PS, Xbox, ...)",20.0,I play for fun,having fun,NaN,NaN,1.0,1.0,Male,19,Employed,High school diploma (or equivalent),USA,South Korea,Reddit,Multiplayer - online - with strangers,14,13.0


In [41]:
print(y.unique())
print(y.describe())

[ 1  8  0 14 12 10 19  3  2  4 15  5  6  7 13 11  9 18 16 21 17 20]
count    12813.000000
mean         5.203153
std          4.703894
min          0.000000
25%          2.000000
50%          4.000000
75%          8.000000
max         21.000000
Name: GAD_T, dtype: float64


In [42]:
print(X.describe())

              Hours  highestleague       streams    Narcissism           Age  \
count  12785.000000            0.0  12720.000000  12802.000000  12813.000000   
mean      21.577552            NaN     10.569654      2.028980     20.959494   
std       14.141691            NaN     11.074622      1.061511      3.302994   
min        0.000000            NaN      0.000000      1.000000     18.000000   
25%       12.000000            NaN      4.000000      1.000000     18.000000   
50%       20.000000            NaN      8.000000      2.000000     20.000000   
75%       28.000000            NaN     15.000000      3.000000     22.000000   
max      420.000000            NaN    420.000000      5.000000     56.000000   

              SWL_T        SPIN_T  
count  12813.000000  12813.000000  
mean      19.780223     19.844767  
std        7.233242     13.461298  
min        5.000000      0.000000  
25%       14.000000      9.000000  
50%       20.000000     17.000000  
75%       26.000000     28.

In [43]:
print(X["highestleague"].isnull().sum())
print(len(X))

12813
12813


In [44]:
X["League"].value_counts(dropna=False)

League
NaN                         1781
Gold                         920
Silver                       625
Platinum                     605
Diamond                      519
                            ... 
EU: gold NA: Platinum          1
BRONZE 1 BIIITCH               1
Silver.                        1
Unranked (former gold V)       1
AHGL                           1
Name: count, Length: 1400, dtype: int64

In [45]:
X["League"].value_counts(dropna=False).head(15)

League
NaN          1781
Gold          920
Silver        625
Platinum      605
Diamond       519
gold          302
Unranked      257
Diamond 5     205
silver        199
Gold V        198
Gold 3        191
Gold 1        190
Silver 1      188
Gold 5        180
Silver 2      169
Name: count, dtype: int64

<span style="color:green"> Group each division of each league into single league group. </span>

In [46]:
X["League"] = X["League"].str.lower()

X["League"] = np.select(
    [
        X["League"].str.contains("bronze", na=False),
        X["League"].str.contains("silver", na=False),
        X["League"].str.contains("gold", na=False),
        X["League"].str.contains("plat", na=False),     
        X["League"].str.contains("diamond", na=False),
        X["League"].str.contains("master", na=False),
        X["League"].str.contains("challenger", na=False),
        X["League"].str.contains("unrank", na=False),
        X["League"].isna()
    ],
    [
        "Bronze",
        "Silver",
        "Gold",
        "Platinum",
        "Diamond",
        "Master",
        "Challenger",
        "Unranked",
        "Unknown"
    ],
    default="Other"
)

In [47]:
X["League"].value_counts().head(20)

League
Gold          3072
Platinum      2553
Silver        2212
Unknown       1781
Diamond       1516
Bronze         515
Other          480
Unranked       434
Master         188
Challenger      62
Name: count, dtype: int64

<span style="color:green"> 100% of highestleague column seems to be missing so we remove it. </span>

In [48]:
X = X.drop(columns=["highestleague"])

In [49]:
X.isnull().sum().sort_values(ascending=False)

Degree        1484
streams         93
Work            31
Hours           28
Reference       11
Narcissism      11
Game             0
SWL_T            0
Playstyle        0
Residence        0
Birthplace       0
Age              0
Platform         0
Gender           0
League           0
whyplay          0
earnings         0
SPIN_T           0
dtype: int64

In [50]:
X["Game"].nunique()

11

In [51]:
X["Game"].value_counts().head(20)

Game
League of Legends      10772
Other                    972
Starcraft 2              325
Counter Strike           300
World of Warcraft        149
Hearthstone               95
Diablo 3                  83
Heroes of the Storm       40
Guild Wars 2              36
Skyrim                    23
Destiny                   18
Name: count, dtype: int64

In [52]:
X["Birthplace"].nunique()

126

In [53]:
X["Residence"].nunique()

109

<span style="color:green"> Replace na with Unknown for Degree, Work, and Reference. </span>

In [54]:
X["Degree"] = X["Degree"].fillna("Unknown")
X["Work"] = X["Work"].fillna("Unknown")
X["Reference"] = X["Reference"].fillna("Unknown")

In [55]:
X.head(20)

,Game,Platform,Hours,earnings,whyplay,League,streams,Narcissism,Gender,Age,Work,Degree,Birthplace,Residence,Reference,Playstyle,SWL_T,SPIN_T
0,Skyrim,"Console (PS, Xbox, ...)",15.0,I play for fun,having fun,Unknown,0.0,1.0,Male,25,Unemployed / between jobs,Bachelor (or equivalent),USA,USA,Reddit,Singleplayer,23,5.0
1,Other,PC,8.0,I play for fun,having fun,Unknown,2.0,1.0,Male,41,Unemployed / between jobs,Bachelor (or equivalent),USA,USA,Reddit,Multiplayer - online - with strangers,16,33.0
2,Other,PC,0.0,I play for fun,having fun,Unknown,0.0,4.0,Female,32,Employed,Bachelor (or equivalent),Germany,Germany,Reddit,Singleplayer,17,31.0
3,Other,PC,20.0,I play for fun,improving,Unknown,5.0,2.0,Male,28,Employed,Bachelor (or equivalent),USA,USA,Reddit,Multiplayer - online - with online acquaintanc...,17,11.0
4,Other,"Console (PS, Xbox, ...)",20.0,I play for fun,having fun,Unknown,1.0,1.0,Male,19,Employed,High school diploma (or equivalent),USA,South Korea,Reddit,Multiplayer - online - with strangers,14,13.0
5,Other,"Console (PS, Xbox, ...)",4.0,I play for fun,relaxing,Other,0.0,2.0,Male,24,Employed,Bachelor (or equivalent),USA,USA,Reddit,Multiplayer - online - with real life friends,17,13.0
6,Other,PC,30.0,I play for fun,relaxing,Unknown,8.0,2.0,Male,29,Employed,High school diploma (or equivalent),USA,USA,Reddit,Multiplayer - online - with online acquaintanc...,16,26.0
8,Other,"Console (PS, Xbox, ...)",2.0,I play for fun,winning,Unknown,0.0,1.0,Female,23,Employed,Bachelor (or equivalent),USA,USA,Reddit,Multiplayer - online - with strangers,12,55.0
9,World of Warcraft,PC,25.0,I play for fun,improving,Other,0.0,1.0,Female,27,Employed,High school diploma (or equivalent),Finland,Finland,Reddit,Multiplayer - online - with online acquaintanc...,13,26.0
10,Other,PC,14.0,I play for fun,having fun,Unknown,0.0,1.0,Female,21,Student at college / university,High school diploma (or equivalent),USA,USA,Reddit,Singleplayer,27,6.0


## Modeling